### **Installations & Imports**

##### **Library & Package Imports**

In [ ]:
import os
import json

import torch
from transformers import pipeline

!pip install gensim
import gensim
from gensim import corpora
from gensim.models import Phrases, Word2Vec
import gensim.downloader as gensim_api

!pip install sklearn
from sklearn.decomposition import PCA

!pip install matplotlib
import matplotlib.pyplot as plt

!pip install wordcloud
from wordcloud import WordCloud

!python -m spacy download en_core_web_lg
import spacy # Note: packaging version 24.1 is installed, not the latest 26.0
from spacy.lang.en.stop_words import STOP_WORDS

##### **Ollama Specific Imports**

In [ ]:
### Ollama Specific Imports

!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

!ollama pull pilardi/sentiment-analysis:llama3
!pip install ollama

from ollama import chat
from pydantic import BaseModel

### **File Reading & Writing**

##### **Reading from article content text files**

In [ ]:
def read_text_file (file_path : str):
    """
    This function returns the contents of a text file as a string.

    :file_path: The file path of the text file.

    Returns the file text as a string, or None if the file could not be opened.
    """
    text = None
    if os.path.isfile(file_path) == False or len(file_path) <= 4 or file_path[-4:] != ".txt":
        print(f"'{file_path}' is either not a text file, or does not exist.")
    else:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()
                # Using the 'with' keyword removes the need to manually close the file
        except:
            # If the file could not be opened/read
            print(f"The file at '{file_path}' could not be read.")
            text = None

    return text

##### **Reading from (article content analysis) results JSON files**

In [ ]:
def read_results_file (file_path : str):
    """
    This function returns the contents of a JSON file.

    :file_path: The file path of the JSON file to read from.

    Returns the JSON data as a dictionary, or None if the file could not be opened.
    """
    data = None
    if os.path.isfile(file_path) == False or len(file_path) <= 5 or file_path[-5:] != ".json":
        print(f"'{file_path}' is either not a JSON file, or does not exist.")
    else:
        try:
            with open(file_path, 'r', encoding="utf-8") as file:
                data = json.load(file)
        except Exception as e:
            print(f"The file at '{file_path}' could not be read. Ensure that it contains valid JSON.")
            data = None

    return data


##### **Writing the (article content analysis) results to a JSON file**

In [ ]:
def save_results (file_path : str, results_data : dict, results_name : str):
    """
    This function saves the output of the full_nlp_analysis() function to a JSON file.

    :file_path: The file path of the JSON file to write to. If no such file exists, it will be created automatically.
    :results_data: A dictionary containing the results data outputted from the full_nlp_analysis() function.
    :results_name: A string denoting the name (dictionary key) that the results data should be stored under.

    Returns True if the results data was saved successfully, and False otherwise.
    """
    if len(file_path) <= 5 or file_path[-5:] != ".json":
        print(f"'{file_path}' is not the file path for a JSON file.")
        return False

    file_data = read_results_file(file_path)
    if file_data == None:
        file_data = {}

    if type(results_data) != dict or type(results_name) != str:
        print("Error: results_data must be a dictionary, and results_name must be a string.")
        return False

    file_data[results_name] = results_data

    with open(file_path, "w") as file:
        json.dump(file_data, file, indent=4, ensure_ascii=False)

    return True


### **Text Pre-Processing**

##### **Pre-processing for sentiment analysis**

In [ ]:
def text_paragraph_split(text : str, min_length : int = 1, max_length = 10000):
    """
    This function splits up text (within a string) into paragraphs.
    Newline characters (\n) are assumed to be the delimiter between paragraphs.

    :text: The string to be split into paragraphs.
    :min_length: An integer denoting the minimum number of characters that a paragraph should contain.
    Paragraphs will be joined together if they are too short individually. The default value is 1.
    :max_length: An integer denoting the maximum number of characters that a paragraph can contain.
    A paragraph will be split up if it is too long. The default value is 10000.

    Returns of list of strings containing the (individual or joined) paragraphs.
    """
    processed_text = [""]
    text = text.replace("=", "")

    if min_length < 1:
        min_length = 1 # Must be at least 1
    if max_length <= min_length:
        max_length = min_length + 1 # Max length must be greater than min length

    for paragraph in text.split("\n"):
        if len(paragraph) > max_length:
            # If the current paragraph alone is too long
            mini = [""]
            for word in paragraph.split(" "):
                if len(mini[-1] + " " + word) > max_length:
                    mini.append(word)
                else:
                    mini[-1] += " " + word
            for sub_paragraph in mini:
                processed_text.append(sub_paragraph)
        elif len(processed_text[-1]) < min_length and not len(processed_text[-1] + " " + paragraph) > max_length:
            # If the previous paragraph is too short, and the combined paragraph will not be too long
            processed_text[-1] += " " + paragraph
        else:
            processed_text.append(paragraph)

    return processed_text

##### **Pre-processing for topic modelling**

In [ ]:
# Date-related words are, in most contexts, not useful for topic modelling
date_blacklist = [
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "isbn", "now", "later", "before", "after", "years", "year", "month", "week", "day", "days"
]

def topmod_pre_process(text : str, min_length : int = 1):
    """
    This function uses spaCy to pre-process English text for topic modelling.
    It removes any stopwords, numbers and punctuation, and splits the text up into individual or joined sentences.

    :text: The string containing the text to pre-process.
    :min_length: An integer denoting the minimum number of characters that a set of joined sentences should contain.
    The default value is 1, but a larger value may result in shorter sentences being joined together.

    Returns of list of strings containing the (individual or joined) sentences.
    """
    processed_text = [""]
    text = text.replace("=", "").replace("\n", " ")
    spacy_model = spacy.load('en_core_web_lg')
    doc = spacy_model(text) # Converts the text into a spaCy doc object

    if min_length < 1:
        min_length = 1 # Must be at least 1

    for sentence in doc.sents:
        processed_sentence = []
        for token in sentence:
            # Iterates over every token (word) in the text
            if token.is_stop or token.is_digit or token.is_punct or token.text.lower() in date_blacklist or not token.text[0].isalnum() or len(token.text) < 3:
                # Exclude any tokens that are stopwords, numbers, punctuation, initials, or date-related
                pass
            else:
                # Include all other tokens, in lowercase form (unless the token is fully uppercase, such as with abbreviations)
                if token.text == token.text.upper():
                    processed_sentence.append(token.text)
                else:
                    processed_sentence.append(token.text.lower())

        joined_sentence = " ".join(processed_sentence)
        if len(processed_text[-1]) < min_length:
            processed_text[-1] += " " + joined_sentence
        else:
            processed_text.append(joined_sentence)

    return processed_text

### **NLP Sentiment Analysis**

##### **Paragraph-level sentiment analysis (using HuggingFace model)**

In [ ]:
# This sentiment analysis model was taken from HuggingFace: https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment
# IMPORTANT: This PLSA model only accepts strings of up to 512 characters in length.
# If using the text_paragraph_split function, set the max_length parameter to 500 to avoid issues.

model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
paragraph_level_sentiment_analysis = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path)


def interpret_plsa_scores(score_list : list, print_scores : bool):
    """
    This function takes in a list of scores generated by the paragraph_level_sentiment_analysis model,
    and outputs two sets of mean and median scores (raw and confidence-adjusted).

    :score_list: A list containing one or more dictionaries in the format {"label":s, "score":x}.
    :print_scores: If set to True, then the four scores will be printed.

    Returns a dictionary containing the mean and median sentiment scores (raw and confidence-adjusted).
    """
    mean = mean_confidence = median = median_confidence = 0 # Initialises four variables with a value of 0
    ssl = []
    csl = []

    for score in score_list:
        if score['label'] == 'positive':
            ssl.append(1)
            csl.append(round(score['score'], 4))
        elif score['label'] == 'neutral':
            ssl.append(0)
            csl.append(0)
        elif score['label'] == 'negative':
            ssl.append(-1)
            csl.append(round(score['score'] * -1, 4))

    ssl.sort()
    csl.sort()
    sl_len = len(score_list)

    if sl_len == 0:
        return {"mean_raw" : 0.0, "mean_con" : 0.0, "median_raw" : 0.0, "median_con" : 0.0}

    mean = round( sum(ssl) / sl_len, 4)
    mean_confidence = round( sum(csl) / sl_len, 4)
    median = round( (ssl[(sl_len-1) // 2] + ssl[round((sl_len-1) / 2)]) / 2 , 4)
    median_confidence = round( (csl[(sl_len-1) // 2] + csl[round((sl_len-1) / 2)]) / 2 , 4)

    if print_scores == True:
        print(f"Mean (raw) = {mean}, Mean (confidence-adjusted) = {mean_confidence}, Median (raw) = {median}, Median (confidence-adjusted) = {median_confidence}.")

    return {"mean_raw" : mean, "mean_con" : mean_confidence, "median_raw" : median, "median_con" : median_confidence}

##### **Aspect-based sentiment analysis (using Ollama model)**

In [ ]:
# This sentiment analysis model was taken from Ollama
# https://ollama.com/pilardi/sentiment-analysis:llama3


class Sentiment_Response (BaseModel):
    sentiment_score : float
    confidence_score : float

def aspect_based_sentiment_analysis(text : list, aspect : str):
    """
    This function uses Ollama and the 'sentiment-analysis:phi3' model to provide aspect-based sentiment analysis.

    :text: A list of text strings to perform sentiment analysis one
    :aspect: The aspect of the text to analyse the sentiment of.

    Returns a list of dictionaries containing the sentiment score and confidence score for each string.
    """
    scores = []

    for chunk in text:
        response = chat(
            model = 'pilardi/sentiment-analysis:llama3',
            format = Sentiment_Response.model_json_schema(),
            messages = [
                {'role' : 'system', 'content' : f"""You are a sentiment analysis model.
                    Your task is to analyse sentiment toward the aspect: {aspect}.
                    Give your response in the format: Sentiment Score: x.xx , Confidence Score: x.xx
                    Provide only one aspect-based sentiment score and confidence score for the following text."""},
                {'role': 'user', 'content': chunk} ],
            options = {'temperature': 0.0 },
            think=False
        )
        score = Sentiment_Response.model_validate_json(response.message.content)
        scores.append(json.loads(response.message.content))

    return scores


def interpret_absa_scores(score_list : list, print_scores : bool):
    """
    This function takes in a list of scores generated by the aspect_based_sentiment_analysis() function,
    and outputs two sets of mean and median scores (raw and confidence-adjusted).

    :score_list: A list containing one or more dictionaries in the format {"sentiment_score":x, "confidence_score":y}.
    :print_scores: If set to True, then the four scores will be printed.

    Returns a dictionary containing the mean and median sentiment scores (raw and confidence-adjusted).
    """
    mean = mean_confidence = median = median_confidence = 0 # Initialises four variables with a value of 0

    ssl = sorted([score["sentiment_score"] for score in score_list]) # List of sentiment scores
    csl = sorted([score["sentiment_score"]*score["confidence_score"] for score in score_list]) # List of sentiment*confidence scores
    sl_len = len(score_list)

    if sl_len == 0:
        return {"mean_raw" : 0.0, "mean_con" : 0.0, "median_raw" : 0.0, "median_con" : 0.0}

    mean = round( sum(ssl) / sl_len, 4)
    mean_confidence = round( sum(csl) / sl_len, 4)
    median = round( (ssl[(sl_len-1) // 2] + ssl[round((sl_len-1) / 2)]) / 2 , 4)
    median_confidence = round( (csl[(sl_len-1) // 2] + csl[round((sl_len-1) / 2)]) / 2 , 4)

    if print_scores == True:
        print(f"Mean (raw) = {mean}, Mean (confidence-adjusted) = {mean_confidence}, Median (raw) = {median}, Median (confidence-adjusted) = {median_confidence}.")

    return {"mean_raw" : mean, "mean_con" : mean_confidence, "median_raw" : median, "median_con" : median_confidence}

##### **Finding the most polarised paragraphs within an article**

In [ ]:
def most_polarised_paragraphs(article_text : str):
    """
    This function uses sentiment analysis to identify the most positive and negative paragraphs within a Wikipedia article.

    :article_text: A string containing the plaintext content of the article.

    Returns a tuple containing the most positive and negative paragraphs (respectively) from the article.
    """
    paragraphs = text_paragraph_split( article_text, max_length=500 )

    plsa = paragraph_level_sentiment_analysis( paragraphs )

    neg_conf, neg_index, pos_conf, pos_index = 0, 0, 0, 0
    index = 0
    for score in plsa:
        if score['label'] == 'positive' and score['score'] > pos_conf:
            pos_index = index
            pos_conf = score['score']
        elif score['label'] == 'negative' and score['score'] > neg_conf:
            neg_index = index
            neg_conf = score['score']
        index += 1

    return (paragraphs[pos_index], paragraphs[neg_index])

### **NLP Topic Identification/Modelling**

##### **Topic Identification using Wikiepdia headings**

In [ ]:
def identify_heading_topics(article_text : str):
    """
    This function takes the content of a Wikipedia article and returns its section headings.

    :article_text: A string containing the plaintext content of the article.

    Returns a tuple of three lists containing the article's main headings, sub-headings, and mini-headings, respectively.
    """
    headers = [[],[],[]]
    equals = ("==", "===", "====")

    for index in [2,1,0]:
        # One iteration for each of the three heading sizes/types
        text_split = article_text.split(equals[index])

        add = False
        for x in text_split:
            if add == False:
                add = True
            else:
                if x.strip() != "" and len(x) < 150:
                    headers[index].append(x.strip().replace("=", ""))
                add = False
        article_text.replace(equals[index], "")

        # Ensure that headings are not duplicated
        headers[0] = [h for h in headers[0] if (h not in headers[1] and h not in headers[2])]
        headers[1] = [h for h in headers[1] if (h not in headers[2])]

    # Remove the 'appendix' headings that can be found in most articles
    for i in range(len(headers[0])):
        if headers[0][i].lower() in ["see also", "notes", "references", "external links"]:
            headers[0] = headers[0][0:i]
            break

    return headers

##### **Topic Modelling using Gensim**

In [ ]:
def lda_topic_modelling(doc_list : list):
    """
    This function performs LDA Topic Modelling on a list of documents (text strings).

    :doc_list: A list of strings containing the sentences/documents to perform topic modelling on.
    The topmod_pre_process() function should be used to produce this.

    Returns a dictionary containing the most prominent topics derived from the input documents.
    A 'probability score' is given for each topic.
    """
    # Decompose each document (sentence string) into a list of words
    sentence_words = [doc.split() for doc in doc_list]

    # Identify meaningful bigrams within the documnets
    bigrams = Phrases(sentence_words, min_count=5, threshold=5, delimiter=' ').export_phrases()

    # Add the bigrams into the decomposed document lists
    for sentence in sentence_words:
        bigram_words = []
        for bigram in bigrams:
            if all(w in sentence for w in bigram.split(" ")):
                sentence.append(bigram)
                bigram_words.extend(bigram.split(" "))
        # Remove the bigrams' constituent words
        for bw in set(bigram_words):
            sentence.remove(bw)

    topic_dict = {}

    # Run the LDA model 5 times
    # The average probability score will be taken for each topic, to account for variance between LDA runs

    for n in range(5):
        # Create a dictionary representation of the words
        dictionary = corpora.Dictionary(sentence_words)
        # Filter out words that occur in less than 3 or more than 80% of the documents.
        dictionary.filter_extremes(no_below=3, no_above=0.8)
        # Convert dictionary to a bag-of-words corpus
        corpus = [dictionary.doc2bow(word) for word in sentence_words]

        # Apply LDA model
        lda_model = gensim.models.ldamodel.LdaModel(
            corpus,
            num_topics=8,
            id2word=dictionary,
            passes=25
        )

        # Generate the most prominent topics and their relative scores.
        topics = lda_model.top_topics(corpus, topn=8)

        # Convert to a dictionary format
        a = 6 if len(topics[0][0]) >= 6 else len(topics[0][0]) # Prevent index out of range errors

        for x in range(a):
            for y in range(8):
                if topics[y][0][x][1] not in topic_dict:
                    topic_dict[ topics[y][0][x][1] ] = topics[y][0][x][0]
                else:
                    topic_dict[ topics[y][0][x][1] ] += topics[y][0][x][0]

    # Sort each topic in the dictionary by its score
    topic_dict = dict(sorted(topic_dict.items(), key=lambda item: item[1], reverse=True))
    for topic in topic_dict:
        topic_dict[topic] = round(topic_dict[topic].item() / 5, 4) # Divide each score by 5 and round it

    topic_dict = dict(list(topic_dict.items())[:40])

    return topic_dict

### **Results Generation**

##### **Master NLP analysis function**

In [ ]:
def full_nlp_analysis(article_content_file, translated_content_file, custom_aspects = None, print_everything = False):
    """
    This function performs sentimnt analysis (paragraph-level + aspect-based) and topic modelling/identification
    on the content of a Wikipedia article. While the sentiment analysis features are comptatible with any language edition,
    an English translation of the article is needed for topic modelling.

    :article_content_file: The name/path of the text file containing the plaintext article content in its orginal language.
    :translated_content_file: The name/path of the text file containing the article content translated into English.
    If the article is already from the English edition, then use the same file as before.
    :custom_aspects: A list of strings containing aspects/topics to perform aspect-based sentiment analysis on.
    If set to None (or an empty list), the top 15 topics generated from topic modelling will be used instead.
    :print_everything: If set to True, then the output of each stage of this function will be printed.

    Returns a dictionary with the following values in relation to the given article:
    'article_length' (int) - The length of the article plaintext in characters.
    'plsa_scores' (dict) - The mean & median paragraph-level sentiment analysis scores.
    'absa_scores' (dict) - The mean & median aspect-based sentiment analysis scores for each aspect.
    'headings' (list) - The article headings.
    'subheadings' (list) - The article subheadings.
    'lda_topics' (dict) - The topics identified by LDA topic modelling, & the associated probability score for each.
    'topic_sentiment' (dict) - The mean & median sentiment scores for the LDA topics.
    'heading_sentiment' (dict) - The mean & median sentiment scores for the headings & subheadings.
    """

    # Read the article content (into a string)
    article_content = read_text_file(article_content_file)
    translated_content = read_text_file(translated_content_file)
    article_char_length = len(article_content)
    if print_everything: print(f"Article is {article_char_length} characters in length.\n")

    # Split the content into paragraphs (remove minimum paragraph/chunk limit if the article is a lot shorter)
    if article_char_length > 20000:
        paragraphs = text_paragraph_split( article_content, min_length=250, max_length=500 )
    else:
        paragraphs = text_paragraph_split( article_content, max_length=500 )
    if print_everything: print(f"Article split into {len(paragraphs)} paragraphs for paragraph-level sentiment analysis.")

    # Perform paragraph-level sentiment analysis
    plsa = paragraph_level_sentiment_analysis( paragraphs )
    if print_everything: print(f"Paragraph-level sentiment analysis scores:")
    plsa_scores = interpret_plsa_scores( plsa, print_everything )

    # Get the article headings
    headings = identify_heading_topics( translated_content )
    if print_everything: print(f"\nArticle headings & subheadings: {headings}")

    # Perform LDA topic modelling
    if article_char_length <= 30000:
        pre_topics = topmod_pre_process( translated_content, article_char_length // 100 )
    else:
        pre_topics = topmod_pre_process( translated_content, 300 )
    lda_topics = lda_topic_modelling( pre_topics )
    if print_everything: print(f"LDA Topics (probability): {lda_topics}")

    # Perform aspect-based sentiment analysis
    aspects = custom_aspects
    if aspects == None or aspects == [] or type(aspects) != list:
        aspects = list( lda_topics.keys() )[:15] # Use top 15 LDA topics
    absa_scores = {}
    if article_char_length < 5000:
        aspect_paragraphs = text_paragraph_split( article_content, article_char_length // 4 )
    else:
        aspect_paragraphs = text_paragraph_split( article_content, article_char_length // 10 )
    if print_everything: print(f"\nArticle split into {len( aspect_paragraphs )} chunks for aspect-based sentiment analysis.")

    for aspect in aspects:
        aspect_sentiment = aspect_based_sentiment_analysis( aspect_paragraphs, aspect )
        if print_everything: print(f"Aspect: '{aspect}'")
        absa_scores[aspect] = interpret_absa_scores( aspect_sentiment, print_everything )

    # Get the sentiment of the topics & headings
    topic_ss = paragraph_level_sentiment_analysis(list(lda_topics.keys()))
    if len(headings[0] + headings[1] + headings[2]) == 0:
        heading_ss = []
    else:
        heading_ss = paragraph_level_sentiment_analysis(headings[0] + headings[1] + headings[2])

    if print_everything: print(f"\nLDA topics sentiment scores:")
    topic_sentiment = interpret_plsa_scores( topic_ss, print_everything )
    if print_everything: print(f"Article heading sentiment scores:")
    heading_sentiment = interpret_plsa_scores( heading_ss, print_everything )

    # Return the results
    results_dict = {
        "article_length" : article_char_length,
        "plsa_scores" : plsa_scores,
        "absa_scores" : absa_scores,
        "headings" : headings[0],
        "subheadings" : headings[1] + headings[2],
        "lda_topics" : lda_topics,
        "topic_sentiment" : topic_sentiment,
        "heading_sentiment" : heading_sentiment
    }

    return results_dict

### **Results Visualisation**

##### **Visualising the extracted topics (using a scatter plot of their vector embeddings)**

In [ ]:
word_vector_model = gensim_api.load("glove-wiki-gigaword-300")
# Alternate word embedding models include:
# "word2vec-google-news-300" - Higher quality, but around 1.5GB in size
# "glove-wiki-gigaword-100" - Around 130MB in size, but lower quality


def topic_vector_embedding(results : dict, title_prefix : str, limit : int = 15):
    """
    This function generates a 2D scatterplot of vector embeddings for the topics extracted from Wikipedia article editions.
    Note that not every topic will have a vector embedding. Multi-word topics (bigrams) will be split into individual words.

    :results: A dictionary containing the NLP analysis results for the article.
    This is expected to be the ouput of the full_nlp_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the scatterplot
    :limit: An integer denoting the maximum number of topics that should be displayed for each edition.
    """
    rl = len(results)
    if type(results) != dict or rl == 0:
        print("The given results are invalid.")
        return

    topics_lists = [list(results[edition]["lda_topics"]) for edition in results]

    # Limit the number of topics displayed for each edition
    for t in range(rl):
        if len(topics_lists[t]) > limit:
            topics_lists[t] = topics_lists[t][:limit]

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    # Create a Matplotlib figure (graph)
    plt.figure(figsize=(11, 6))
    plt.grid()

    vectors = dict()
    cleaned_words = []

    # Iterate over a maximum of 10 lists of topics
    for i in range(rl if rl <= 10 else 10):
        cleaned_words.append([])
        # Select only the topics/words that have vector embeddings available
        for word in topics_lists[i]:
            if type(word) == str and " " in word:
                for w in word.split(" "):
                    if w not in topics_lists[i] and w in word_vector_model:
                        cleaned_words[i].append(w)
            elif type(word) == str and word in word_vector_model:
                cleaned_words[i].append(word)

        # Get the vector embedding for each topic
        for word in cleaned_words[i]:
            vectors[word] = word_vector_model[word]


    # Fit the Principal Component Analysis (PCA) model using all the topic vectors at the same time
    pca = PCA(n_components=2).fit(list(vectors.values()))
    # Use Principal Component Analysis (PCA) to reduce the dimensions of the embeddings from 300 to 2
    pca_vectors = pca.transform(list(vectors.values()))

    for i in range(rl if rl <= 10 else 10):
        # Plot the topic vector embeddings as points on a scatter graph
        for cw in cleaned_words[i]:
            vector_index = list(vectors.keys()).index(cw)
            if cleaned_words[i].index(cw) != 0:
                plt.scatter(pca_vectors[vector_index, 0], pca_vectors[vector_index, 1], color=color_list[i], alpha=0.5)
            else:
                plt.scatter(pca_vectors[vector_index, 0], pca_vectors[vector_index, 1], color=color_list[i], alpha=0.5, label=list(results)[i])
            # Annotate the points with the respective topics
            plt.annotate(cw, xy=(pca_vectors[vector_index, 0] + 0.08, pca_vectors[vector_index, 1] + 0.08), fontsize=10, fontweight=540)

    # Plot the graph labels and show the graph
    plt.title(f"{title_prefix} - Topic Vector Embeddings", fontsize=14, fontweight=600)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.legend()
    plt.show()

##### **Visualising the extracted topics (using word clouds)**

In [ ]:
def topics_word_cloud(topics_dict : dict, title_prefix : str, exclude : list = None):
    """
    This function takes a dictionary of topics and generates a Word Cloud image.

    :topics_dict: A dictionary of topics and their prominence scores.
    This is expected to be the output of lda_topic_modelling() (or 'lda_topcis' from the output of full_nlp_analysis()).
    :title_prefix: A string containing the prefix of the title to be displayed above the word cloud image.
    :exclude: Topics containing any of the strings in this list will not be included in the woud cloud. Case insensitive.
    """
    if type(exclude) == list:
        remove_topics = []
        for topic in topics_dict:
            for ex in exclude:
                if ex.lower() in topic.lower():
                    remove_topics.append(topic)
                    break
        for rt in remove_topics:
            topics_dict.pop(rt)

    word_cloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(topics_dict)

    plt.figure(figsize=(12, 6))
    plt.imshow(word_cloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"{title_prefix} - Topic Word Cloud", fontsize=14, fontweight=600)
    plt.show()

##### **Visualising the aspect-based sentiment analysis scores (using a line chart)**

In [ ]:
def absa_line_chart(results : dict, title_prefix : str, which_score : str = "mean_con"):
    """
    This function generates a line chart of aspect-based sentiment analysis scores for one or more Wikipedia article editions.

    :results: A dictionary containing the NLP analysis results for the article.
    This is expected to be the ouput of the full_nlp_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the line chart.
    :which_score: A string denoting which sentiment scores to use.
    The options are "mean_raw", "mean_con", "median_raw", and "median_con". The default is "mean_con".
    """
    rl = len(results)
    if type(results) != dict or rl == 0:
        print("The given results are invalid.")
        return

    aspects = list(results[list(results)[0]]["absa_scores"])
    for edition in results:
        if list(results[edition]["absa_scores"]) != aspects:
            # This loop checks that each set of scores (edition) has the same aspects.
            print("The given scores_list is not valid - every dictionary in the list must have the same keys (aspects).")
            return

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    if which_score not in ["mean_raw", "mean_con", "median_raw", "median_con"]:
        which_score = "mean_con"

    plt.figure(figsize=(12, 5))

    for i in range(rl):
        scores = [results[list(results)[i]]["absa_scores"][aspect][which_score] for aspect in aspects]
        plt.plot(aspects, scores,  marker='o', linestyle='-', label=list(results)[i], color=color_list[i], alpha=0.5)

    plt.grid(True)
    plt.xlabel("Aspect", fontsize=11, fontweight=600)
    plt.ylabel(f"Score ({which_score})", fontsize=11, fontweight=600)
    plt.xticks(rotation=45)
    plt.title(f"{title_prefix} - Aspect-Based Sentiment Analysis Scores", fontsize=14, fontweight=600)
    plt.legend()
    plt.show()

##### **Visualising the other sentiment scores (using a bar chart)**

In [ ]:
def sentiment_bar_chart(results : dict, title_prefix : str, which_score : str = "mean_con"):
    """
    This function generates a bar chart of sentiment scores for one or more Wikipedia article editions.
    This includes the paragraph-level sentiment analysis scores, plus the topic and heading sentiment scores.

    :results: A dictionary containing the NLP analysis results for the article.
    This is expected to be the ouput of the full_nlp_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the bar chart.
    :which_score: A string denoting which sentiment scores to use.
    The options are "mean_raw", "mean_con", "median_raw", and "median_con". The default is "mean_con".
    """
    rl = len(results)
    if type(results) != dict or len(results) == 0:
        print("The given results are invalid.")
        return

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    plt.figure(figsize=(10, 6))

    x = [0, 1, 2]

    width = 0.8 / rl
    offsets = [n*width for n in range(rl)]
    if rl % 2 == 0:
        mid = (offsets[rl//2] + offsets[(rl//2)-1]) / 2
    else:
        mid = offsets[rl//2]
    offsets = [o - mid for o in offsets]

    for i, edition in enumerate(results):
        y = [results[edition]["plsa_scores"][which_score],
             results[edition]["topic_sentiment"][which_score],
             results[edition]["heading_sentiment"][which_score]]
        plt.bar([n+offsets[i] for n in x], y, width, color=color_list[i], edgecolor = "black", alpha=0.75)

    plt.xticks(x, ['Paragraph\nLevel', 'Topics', 'Headings'])
    plt.xlabel("Category", fontsize=11, fontweight=600)
    plt.ylabel(f"Sentiment Score ({which_score})", fontsize=11, fontweight=600)
    plt.legend(list(results))
    plt.title(f"{title_prefix} - Other Sentiment Scores", fontsize=14, fontweight=600)
    plt.gca().set_axisbelow(True)
    plt.grid(axis = 'y')
    plt.show()

### **Code Execution Area**

##### **Running Topic Modelling (seperately)**

In [ ]:
article_content = read_text_file("Article_Text_Spanish_Translated.txt")

ml = len(article_content) // 100 if len(article_content) <= 30000 else 300
pre_content = topmod_pre_process(article_content, min_length=ml)

topics = lda_topic_modelling(pre_content)

print(f"Topics: {topics}")

##### **Running the master function**

In [ ]:
# Use the following custom list of topics for ABSA
custom_aspects = ['kennedy', 'president', 'united states', 'american', 'berlin', 'soviet', 'administration', 'vietnam', 'cuba', 'assassination', 'life', 'military', 'civil rights']


### For Umm Kulthum's English article
results_1 = full_nlp_analysis("Article_Text_English.txt", "Article_Text_English.txt", custom_aspects, False)
print("English results:")
for key, value in results_1.items():
    print(f"{key} : {value}")


### For Umm Kulthum's Egyptian Arabic article
results_2 = full_nlp_analysis("Article_Text_Russian.txt", "Article_Text_Russian_Translated.txt", custom_aspects, False)
print("\nRussian results:")
for key, value in results_2.items():
    print(f"{key} : {value}")


### For Umm Kulthum's Arabic article
results_3 = full_nlp_analysis("Article_Text_Spanish.txt", "Article_Text_Spanish_Translated.txt", custom_aspects, False)
print("\nSpanish results:")
for key, value in results_3.items():
    print(f"{key} : {value}")


### For Umm Kulthum's Hebrew article
results_4 = full_nlp_analysis("Article_Text_Vietnamese.txt", "Article_Text_Vietnamese_Translated.txt", custom_aspects, False)
print("\nVietnamese results:")
for key, value in results_4.items():
    print(f"{key} : {value}")

##### **Saving the results (as JSON files)**

In [ ]:
save_results("article_content_analysis_results.json", results_1, "English")
save_results("article_content_analysis_results.json", results_2, "Russian")
save_results("article_content_analysis_results.json", results_3, "Spanish")
save_results("article_content_analysis_results.json", results_4, "Vietnamese")

##### **Generating visualisations for the topic modelling results (scatterplot of vector embeddings)**

In [ ]:
results = read_results_file("article_content_analysis_results.json")

topic_vector_embedding(results, "John F. Kennedy")

##### **Generating visualisations for the topic modelling results (word clouds)**

In [ ]:
results = read_results_file("article_content_analysis_results.json")

for edition in results:
    topics_word_cloud(results[edition]["lda_topics"], f"John F. Kennedy ({edition})", ['john', 'kennedy'])
    print()

##### **Generating visualisations of the ABSA scores (line chart)**

In [ ]:
results = read_results_file("article_content_analysis_results.json")

absa_line_chart(results, "John F. Kennedy", "mean_con")

##### **Generating visualisations of the PLSA + topic + heading sentiment scores (bar chart)**

In [ ]:
results = read_results_file("article_content_analysis_results.json")

sentiment_bar_chart(results, "John F. Kennedy", "mean_con")

##### **Printing the most polarised paragraphs within an article edition**

In [ ]:
article_content = read_text_file("Article_Text_English.txt")
polar_paragraphs = most_polarised_paragraphs(article_content)
print(f"Most positive paragraph: {polar_paragraphs[0]}\nMost negative paragraph: {polar_paragraphs[1]}")